# 08 — Mammo-FM Downstream

One directly runnable notebook represents one architecture. Change only CONDITION and SEED to cover its 12 protocol jobs.

## 1. Experiment configuration

In [ ]:
GPU = 0
from pathlib import Path
import json
import sys
ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'configs').is_dir())
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'notebooks/utility'))
from notebooks.utility.downstream_experiment import (
    build_error_case_table, configure_environment, construct_dataset, experiment_configuration,
    load_adapter, load_existing_outputs, load_model, load_prediction_rows, plot_calibration, plot_source_accounting,
    plot_training_history, plot_validation_curves, resume_status, run_validation,
    saved_artifacts, train, training_budget,
)
ARCHITECTURE = 'mammofm'
CONDITION = 'real_only'
SEED = 17
RESUME = True
CONFIRM_EXISTING_OUTPUT = False
RUN_TRAINING = False
RUN_VALIDATION = False
configuration = experiment_configuration(ROOT, ARCHITECTURE, CONDITION, SEED, gpu=GPU, resume=RESUME)
configuration['root'] = str(ROOT)
configuration['confirm_existing_output'] = CONFIRM_EXISTING_OUTPUT
configuration

## 2. Environment and GPU

In [ ]:
environment = configure_environment(configuration)
environment

## 3. Selected generators

In [ ]:
from notebooks.utility.downstream_protocol import load_selected_generators, selection_summary
selected_generators = load_selected_generators(ROOT, required=False)
# Model-free configuration view: selected generator, model/generation identity, FILTERED
# manifest SHA-256, active amendment and benchmark run. No model is loaded here.
selection_configuration = selection_summary(selected_generators)
selection_configuration or 'Not required for real-only conditions; run notebook 06 before a synthetic condition.'

## 4. Dataset construction

In [ ]:
dataset = construct_dataset(ROOT, configuration)
len(dataset['train_rows']), len(dataset['validation_rows'])

## 5. Dataset audit

In [ ]:
dataset['audit']  # fails before training if train/validation patients overlap

## 6. Model loading

In [ ]:
model_bundle = None
if RUN_TRAINING:
    model_bundle = load_model(configuration)
model_bundle if model_bundle is not None else 'Deferred until RUN_TRAINING=True; no weights downloaded during protocol review.'

## 7. Trainable parameters

In [ ]:
trainable_parameters = None
if model_bundle is not None:
    _, model = model_bundle
    trainable_parameters = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
trainable_parameters if trainable_parameters is not None else 'Not yet instantiated'

## 8. Training configuration

In [ ]:
training_budget(configuration)

## 9. Resume status

In [ ]:
resume_information = resume_status(ROOT, configuration)
resume_information

## 10. Training

In [ ]:
training_result = None
if RUN_TRAINING:
    training_result = train(ROOT, configuration, dataset)
else:
    print('Training disabled. Review the audit, then set RUN_TRAINING=True for the selected CONDITION × SEED.')

## 11. Training curves

In [ ]:
existing_outputs = load_existing_outputs(ROOT, configuration)
training_history_figure = plot_training_history(existing_outputs['history']) if existing_outputs['history'] else None
source_accounting_figure = plot_source_accounting(existing_outputs['source_accounting']) if existing_outputs['source_accounting'] else None
{'training_curves': training_history_figure, 'source_sampling_counts': existing_outputs['source_accounting'], 'source_accounting_figure': source_accounting_figure} if training_history_figure else 'Not yet evaluated'

## 12. Best checkpoint

In [ ]:
checkpoint = training_result['checkpoint'] if training_result else existing_outputs['checkpoint']
{'checkpoint': checkpoint, 'criterion': configuration['policy']['checkpoint_criterion'],
 'tie_policy': 'lower validation loss, then earlier epoch'}

## 13. Validation inference

In [ ]:
validation_result = None
if RUN_VALIDATION:
    if checkpoint is None: raise RuntimeError('A validation-selected checkpoint is required')
    validation_result = run_validation(ROOT, configuration, dataset, checkpoint)
validation_rows = validation_result['rows'] if validation_result else existing_outputs['predictions']
validation_metrics = validation_result['metrics'] if validation_result else existing_outputs['validation_metrics']
validation_result or ({'loaded_predictions': len(validation_rows)} if validation_rows else 'Not yet evaluated')

## 14. Validation metrics

In [ ]:
validation_curve_figure = plot_validation_curves(validation_rows, validation_metrics['threshold']) if validation_rows and validation_metrics else None
validation_metrics if validation_metrics else 'Not yet evaluated'

## 15. Calibration

In [ ]:
calibration_figure = plot_calibration(validation_rows) if validation_rows else None
calibration_figure if calibration_figure else 'Not yet evaluated'

## 16. Error analysis

In [ ]:
error_cases = build_error_case_table(validation_rows, validation_metrics['threshold']) if validation_rows and validation_metrics else None
error_tables = ({'false positives': error_cases[error_cases.error_type == 'false_positive'],
                 'false negatives': error_cases[error_cases.error_type == 'false_negative'],
                 'highest-confidence correct predictions': error_cases[error_cases.error_type == 'correct']}
                if error_cases is not None else 'Not yet evaluated')
from notebooks.utility.classifier_interpretability import largest_ft_fs_disagreements
ft_rows = load_prediction_rows(ROOT / 'results/publication_v2/downstream/mammofm/real_plus_best_finetuned_positive' / f'seed_{SEED}/validation_predictions.csv')
fs_rows = load_prediction_rows(ROOT / 'results/publication_v2/downstream/mammofm/real_plus_best_fromscratch_positive' / f'seed_{SEED}/validation_predictions.csv')
largest_disagreements = largest_ft_fs_disagreements(ft_rows, fs_rows) if ft_rows and fs_rows else 'Not yet evaluated'
error_tables = {**error_tables, 'largest FT-vs-FS disagreements': largest_disagreements} if isinstance(error_tables, dict) else error_tables
error_tables

## 17. Interpretability

In [ ]:
from notebooks.utility.classifier_interpretability import mammofm_attribution, preregistered_cases
RUN_INTERPRETABILITY = False
interpretability_manifest = Path(configuration['results_dir']) / 'interpretability_manifest.json'
interpretability_cases = preregistered_cases(validation_rows, validation_metrics['threshold']) if validation_rows and validation_metrics else None
if RUN_INTERPRETABILITY:
    if not checkpoint or not interpretability_cases:
        raise RuntimeError('A validation-selected checkpoint and deterministic validation cases are required')
    adapter = load_adapter(configuration); model = adapter.load_checkpoint(checkpoint)
    case_loader = adapter.build_validation_dataloader(interpretability_cases, seed=SEED)
    attribution_root = Path(configuration['results_dir']) / 'interpretability'; attribution_root.mkdir(parents=True, exist_ok=True)
    import numpy as np
    case_index = 0
    for images, _labels in case_loader:
        for image in images:
            category = interpretability_cases[case_index]['category']; heatmap = mammofm_attribution(model, image.unsqueeze(0))
            np.save(attribution_root / f'{category}_{case_index}.npy', heatmap); case_index += 1
({'method': 'architecture-compatible EfficientNet spatial attribution', 'deterministic_cases': {'TP/TN/FP/FN': interpretability_cases, 'largest FT-vs-FS disagreement': largest_disagreements}, 'saved_manifest': json.loads(interpretability_manifest.read_text()) if interpretability_manifest.is_file() else 'Not yet evaluated'}
 if interpretability_cases else 'Not yet evaluated — Mammo-FM attribution has not been evaluated; required method: architecture-compatible EfficientNet spatial-feature attribution.')

## 18. Saved artifacts

In [ ]:
saved_artifacts(ROOT, configuration)